# 01 – Data Overview & Train/Val/Test Splits

This notebook provides an end-to-end overview of the raw dataset, visualises
the frame distribution across videos, and produces a **group-aware** split
that prevents data leakage between train/val/test sets.

**Output:** `datasets/splits/detection_split.json`

## 1 · Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Make the trash_detection package importable from the repo root
REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from trash_detection.io import find_images, find_frames, list_video_ids
from trash_detection.splits import make_detection_split, save_split, load_split
from trash_detection.viz import show_image_grid, plot_class_distribution
import cv2

# ── paths ────────────────────────────────────────────────────────────────────
RAW_IMAGES_DIR = REPO_ROOT / "datasets" / "raw" / "images"
FRAMES_DIR     = REPO_ROOT / "datasets" / "raw" / "frames"
SPLITS_DIR     = REPO_ROOT / "datasets" / "splits"
SPLIT_PATH     = SPLITS_DIR / "detection_split.json"

print("Repo root:", REPO_ROOT)
print("Raw images dir exists:", RAW_IMAGES_DIR.exists())
print("Frames dir exists     :", FRAMES_DIR.exists())

## 2 · Dataset Structure Overview

We expect two sources of images:

| Source | Path | Group key |
|--------|------|-----------|
| Standalone images | `datasets/raw/images/` | `"standalone"` |
| Video frames | `datasets/raw/frames/<video_id>/` | `<video_id>` |

In [ ]:
# ── Standalone images ─────────────────────────────────────────────────────────
standalone_images = find_images(RAW_IMAGES_DIR) if RAW_IMAGES_DIR.exists() else []
print(f"Standalone images : {len(standalone_images)}")

# ── Video frames ──────────────────────────────────────────────────────────────
video_ids = list_video_ids(FRAMES_DIR) if FRAMES_DIR.exists() else []
print(f"\nVideo IDs found   : {len(video_ids)}")

frame_counts: dict[str, int] = {}
all_frames: list[Path] = []
for vid in video_ids:
    frames = find_images(FRAMES_DIR / vid)
    frame_counts[vid] = len(frames)
    all_frames.extend(frames)
    print(f"  {vid:30s}  {len(frames):>5d} frames")

print(f"\nTotal frame images: {len(all_frames)}")
print(f"Grand total       : {len(standalone_images) + len(all_frames)}")

## 3 · Exploratory Data Analysis

In [ ]:
# Bar chart – frames per video
if frame_counts:
    plot_class_distribution(frame_counts, title="Frames per Video")
else:
    print("No video frames found yet.")

In [ ]:
# Sample grid – first frame from each video (up to 8)
sample_paths = [find_images(FRAMES_DIR / v)[0] for v in video_ids if find_images(FRAMES_DIR / v)][:8]
if sample_paths:
    sample_imgs = [cv2.imread(str(p)) for p in sample_paths]
    show_image_grid(sample_imgs, titles=[p.parent.name for p in sample_paths], cols=4)
else:
    print("No sample frames available yet.")

## 4 · Define Splits

Frames from the **same video** must stay in the same split to avoid leakage
(adjacent frames are visually nearly identical).  We use `make_detection_split`
which groups items by *video_id* before shuffling and splitting.

In [ ]:
# Build parallel lists: item paths + group labels
all_items: list[str] = []
all_groups: list[str] = []

# Standalone images → single group so they stay together (or assign "standalone")
for p in standalone_images:
    all_items.append(str(p))
    all_groups.append("standalone")

# Frame images → group by video_id
for vid in video_ids:
    for p in find_images(FRAMES_DIR / vid):
        all_items.append(str(p))
        all_groups.append(vid)

print(f"Total items : {len(all_items)}")
print(f"Unique groups: {len(set(all_groups))}")

if all_items:
    split = make_detection_split(
        items=all_items,
        groups=all_groups,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
        seed=42,
    )

    save_split(split, SPLIT_PATH)
    print(f"\nSplit saved → {SPLIT_PATH}")
    for name, items in split.items():
        print(f"  {name:5s}: {len(items):>5d} images")
else:
    print("No images found – populate datasets/raw/ first.")
    split = {"train": [], "val": [], "test": []}

## 5 · Verify Split

Check that no video appears in more than one split (leakage check).

In [ ]:
if all_items:
    # Re-load from disk to confirm serialisation round-trip
    split_loaded = load_split(SPLIT_PATH)

    def get_groups_in_split(paths: list[str]) -> set[str]:
        groups_seen = set()
        for p in paths:
            path = Path(p)
            # frames are under frames/<video_id>/frame_*.jpg
            if path.parent.parent.name == "frames":
                groups_seen.add(path.parent.name)
            else:
                groups_seen.add("standalone")
        return groups_seen

    train_groups = get_groups_in_split(split_loaded["train"])
    val_groups   = get_groups_in_split(split_loaded["val"])
    test_groups  = get_groups_in_split(split_loaded["test"])

    print(f"Train groups : {sorted(train_groups)}")
    print(f"Val groups   : {sorted(val_groups)}")
    print(f"Test groups  : {sorted(test_groups)}")

    # Assert no group appears in multiple splits
    tv_overlap = train_groups & val_groups
    tt_overlap = train_groups & test_groups
    vt_overlap = val_groups & test_groups

    assert not tv_overlap, f"LEAKAGE: groups in both train & val: {tv_overlap}"
    assert not tt_overlap, f"LEAKAGE: groups in both train & test: {tt_overlap}"
    assert not vt_overlap, f"LEAKAGE: groups in both val & test: {vt_overlap}"

    total_loaded = sum(len(v) for v in split_loaded.values())
    assert total_loaded == len(all_items), "Item count mismatch after round-trip!"

    print("\n✓ No leakage detected.")
    print(f"✓ Round-trip OK ({total_loaded} items).")
else:
    print("Skipped – no data yet.")